# "Tunnels are fast but sometimes they collapse": Part 1

## Introduction

In the first part of the hackaton you have to find the optimal position for the three detectors. The positions that you find optimal will be used in the next part.

As we discussed in the introductory presentation, the data provided contains tunnel and open sky measurements for a hypothetical large active area, that spans the whole tunnel surface (8 x 20 m). 
However, this is not realistic. Therefore, we will be allowed to use only three detector chamber of 1 square meter each. 
The position of these chambers in the tunnel surface will determine how visible the chamber is in our reconstructed images.

The recommended approach is to write an iterative algorithm that optimizes the detector's position based on a significative loss function.

## Scoring

Participants have to provide the positions of the three detectors:
$$ (X_i, Y_i) \ \ \ \ i=0,1,2 $$
The answers cannot be random and have to be justified by some algorithm that can be reproducible. 

The score of this part will be obtained by the Peak Significance metric. 
This metric is extracted from the transmission coefficient maps, which are obtained by projecting the detected muons to a certain Z plane, and plotting the ratio of tunnel to open sky muons in a 2D histogram. Given such a transmission coefficeint map, $R$, the Peak Significance is computed as:

$$ \text{Score} = \dfrac{\max(R) - \mu(R)}{\sigma(R)}, $$

where $\max(R)$ is the value of the maximum bin in $R$, and $\mu(R)$ and $\sigma(R)$ are the mean and sigma of the bin values of the full map.

In order to homogenize the provided solutions, the ranges for the reconstructed maps must be $-1500$ cm and $1500$ cm, both in X and Y.

### Recommended Python packages:
- Python 3.14
- Numpy
- Pandas
- Matplotlib
- h5py
- PyTorch (can be installed from https://pytorch.org/get-started/locally/, depending on setup and CUDA version)

## Getting the data

The following snippet of code will give you access to the data in case you don't have it for the previous part of the hackaton.

In [ ]:
##### Getting the data
import urllib.request
urllib.request.urlretrieve('https://nextcloud.ifca.es/index.php/s/ZdoGxoT8Fs6NDwe/download', 'opensky.h5')
urllib.request.urlretrieve('https://nextcloud.ifca.es/index.php/s/KXSjiPqybAQjjgC/download', 'tunnel.h5')

print("Data downloaded successfully")

## Import necessary packages

In [ ]:
%pip install numpy pandas matplotlib h5py

In [ ]:
import matplotlib.colors as colors
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd

import h5py

import torch
import torch.nn as nn
import torch.nn.functional as F

## Load the data

The data can be loaded as Pandas DataFrames, as NumPy arrays or using the package h5py.

Since the dataset is quite heavy, for some applications it is useful to load the data as a NumPy memory map. This lowers the RAM used in the execution, avoiding out of memory errors. An example of such is provided in the following code snippet.

In [ ]:
def load_to_memmap(file_list, filename='opensky_combined.dat', dtype=np.float32, N=-1):
    # Determine total dimensions
    shapes = []
    for f in file_list:
        with h5py.File(f, 'r') as h5:
            shape = h5['df/block0_values'].shape
            if shape[0] < shape[1]:
                shape = (shape[1], shape[0])
            shapes.append(shape)

    total_rows = sum(s[0] for s in shapes)
    n_features = shapes[0][1]

    print(f"Data to load: {file_list}")
    print(f"  Total number of rows: {total_rows}")
    print(f"  Number of features:   {n_features}")
    if N>0: print(f"   Will load {N} events")

    # Create memory-mapped file on disk
    if N>0 and total_rows>N:
        mmap_array = np.memmap(filename, dtype=dtype, mode='w+', shape=(N, n_features))
    else:
        mmap_array = np.memmap(filename, dtype=dtype, mode='w+', shape=(total_rows, n_features))

    current_idx = 0
    for f, shape in zip(file_list, shapes):
        with h5py.File(f, 'r') as h5:
            data = h5['df/block0_values'][:]
            if data.shape[0] < data.shape[1]:
                data = data.T
            rows = shape[0]
            if N>0 and current_idx+rows>N:
                mmap_array[current_idx:N] = data[0:N,:].astype(dtype, copy=False)
                current_idx = N
            else:
                mmap_array[current_idx:current_idx + rows] = data.astype(dtype, copy=False)
                current_idx += rows
                
    # Flush changes to disk
    mmap_array.flush()
    return mmap_array

In [ ]:
openskyFiles = ['data/opensky.h5']
tunnelFiles = ['data/tunnel.h5']

openskyfull = load_to_memmap(openskyFiles, 'data/opensky.dat')
tunnelfull  = load_to_memmap(tunnelFiles, 'data/tunnel.dat')

In [ ]:
import pandas as pd
pd.DataFrame(openskyfull)

## Plotting basic variables

In [ ]:
fig, axs = plt.subplots(2, 3, figsize = (12, 8), tight_layout=True)
# We can set the number of bins with the *bins* keyword argument.
n_bins = 100
xrange = (-1000,1000)
axs[0][0].hist(openskyfull[:,0], bins=n_bins, range=xrange, histtype='step');
axs[0][0].set_xlabel("x (cm)")
axs[0][1].hist(openskyfull[:,1], bins=n_bins, range=xrange, histtype='step');
axs[0][1].set_xlabel("y (cm)")
axs[0][2].hist(openskyfull[:,2], bins=n_bins, range=(-330,-329), histtype='step');
axs[0][2].set_xlabel("z (cm)")
axs[1][0].hist(tunnelfull[:,0], bins=n_bins, range=xrange, histtype='step');
axs[1][0].set_xlabel("x (cm)")
axs[1][1].hist(tunnelfull[:,1], bins=n_bins, range=xrange, histtype='step');
axs[1][1].set_xlabel("y (cm)")
axs[1][2].hist(tunnelfull[:,2], bins=n_bins, range=(-330,-329), histtype='step');
axs[1][2].set_xlabel("z (cm)")
plt.savefig("plots/positions.png")

In [ ]:
xrange = [-1.,1.]
fig, axs = plt.subplots(2, 3, figsize = (12, 8), tight_layout=True)
# We can set the number of bins with the *bins* keyword argument.
n_bins = 100
xrange = (-1.,1.)
axs[0][0].hist(openskyfull[:,3], bins=n_bins, range=xrange, histtype='step');
axs[0][0].set_xlabel("vx")
axs[0][1].hist(openskyfull[:,4], bins=n_bins, range=xrange, histtype='step');
axs[0][1].set_xlabel("vy")
axs[0][2].hist(openskyfull[:,5], bins=n_bins, range=xrange, histtype='step');
axs[0][2].set_xlabel("vz")
axs[1][0].hist(tunnelfull[:,3], bins=n_bins, range=xrange, histtype='step');
axs[1][0].set_xlabel("vx")
axs[1][1].hist(tunnelfull[:,4], bins=n_bins, range=xrange, histtype='step');
axs[1][1].set_xlabel("vy")
axs[1][2].hist(tunnelfull[:,5], bins=n_bins, range=xrange, histtype='step');
axs[1][2].set_xlabel("vz")
plt.savefig("plots/directions.png")

## The dataselector class

This class will help you to select the dataset for each of the chambers you have chosen.

Also, a differentiable approach to this is provided. This apporach is based on asigning weigths to the muons based on whether they hit the active area or not.
The weigths are computed using a sigmoid (differentiable).

In [ ]:
import numpy as np

class DatasetSelector:
    def __init__(self, limits, detsize):
        self.detsize = detsize
        self.xmin = limits[0][0]
        self.xmax = limits[0][1]
        self.ymin = limits[1][0]
        self.ymax = limits[1][1]

    def get(self, data):
        preselect = data[(data[:, 0] > self.xmin) & (data[:, 0] < self.xmax) & (data[:,1] > self.ymin) & (data[:,1] < self.ymax)]
        select = preselect[(preselect[:,0] - self.detsize/preselect[:, 5] * preselect[:,3] > self.xmin) &
                           (preselect[:,0] - self.detsize/preselect[:, 5] * preselect[:,3] < self.xmax) &
                           (preselect[:,1] - self.detsize/preselect[:, 5] * preselect[:,4] > self.ymin) &
                           (preselect[:,1] - self.detsize/preselect[:, 5] * preselect[:,4] < self.ymax)]
        return select

In [ ]:
def soft_box_2d(x: torch.Tensor, y: torch.Tensor, center: torch.Tensor, size_xy: float, temperature: float = 1.0) -> torch.Tensor:
    """Computes a smooth differentiable acceptance weight in [0, 1]
    for points (x, y) falling inside the detector box centered at `center`.
    """
    half_w = size_xy / 2.0
    x_min, x_max = center[0] - half_w, center[0] + half_w
    y_min, y_max = center[1] - half_w, center[1] + half_w

    # Smooth step function using sigmoids
    in_x = torch.sigmoid((x - x_min) / temperature) * torch.sigmoid((x_max - x) / temperature)
    in_y = torch.sigmoid((y - y_min) / temperature) * torch.sigmoid((y_max - y) / temperature)
    return in_x * in_y